# 03 Embeddings

## Objective

This notebook converts text chunks into vector embeddings.

## Goals

- Load text chunks
- Generate embeddings
- Analyze embedding dimensions
- Prepare data for vector search

## Use Case

Fraud Detection Knowledge Assistant

In [2]:
!pip install sentence_transformers -q

In [3]:
!pip install pypdf pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 5.9 MB/s eta 0:00:00


In [11]:
import pandas as pd
from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# PDF Text Extraction Function
def extract_pdf_text(pdf_path):
  reader = PdfReader(pdf_path)
  text = ""
  for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
      text += page_text + "\n"
  return text

# Text Chunking Function
def create_chunks(text, chunk_size=1000):
  chunks = []
  for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i + chunk_size])
  return chunks

# Load Documents
RAW_DATA_DIR = Path("data/raw")
pdf_files = list(RAW_DATA_DIR.glob("*.pdf"))
documents = []
for pdf_file in pdf_files:
  text = extract_pdf_text(pdf_file)
  documents.append({"file_name": pdf_file.name, "text": text})
documents_df = pd.DataFrame(documents)

# Create Chunks
chunk_records = []
for _, row in documents_df.iterrows():
  chunks = create_chunks(row["text"])
  for idx, chunk in enumerate(chunks):
    chunk_records.append({"file_name": row["file_name"], "chunk_id": idx, "chunk_text": chunk})
chunks_df = pd.DataFrame(chunk_records)
print(f"Total chunks: {len(chunks_df)}")

# Load Embedding Model
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")

# Generate Embeddings
embeddings = model.encode(chunks_df["chunk_text"].tolist(), show_progress_bar=True)

# Analyze Embeddings
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimensions: {len(embeddings[0])}")

# Preview First Embedding
embeddings[0][:10]

Total chunks: 88


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Number of embeddings: 88
Embedding dimensions: 384


array([-0.081213  , -0.00326374,  0.01820515, -0.08367913,  0.02623537,
       -0.02299135, -0.01265612, -0.01871347,  0.03342513, -0.07637702],
      dtype=float32)

## Conclusion

Text chunks were successfully converted into vector embeddings.

The generated embeddings will be stored in a vector database in the next notebook to enable semantic search.

In [18]:
!git push

Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 885 bytes | 177.00 KiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/5b0chmann/fraud-detection-llmops.git
   00f3f62..6ffd7f2  main -> main
